### Traitement des données : Dans cette partie, il s'agira pour nous de préparer les données pour la modélisation à travers le nettoyage, l'encodage et le feature engineering

In [46]:
# Importating necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder

In [47]:
# Charging the  dataset
data = pd.read_csv("../data/raw/train.csv")
data.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [48]:
# Types of columns in the dataset
data.dtypes

Id                 int64
MSSubClass         int64
MSZoning          object
LotFrontage      float64
LotArea            int64
                  ...   
MoSold             int64
YrSold             int64
SaleType          object
SaleCondition     object
SalePrice          int64
Length: 81, dtype: object

In [49]:
# Let's delete the 'Id' column as it is not useful for our analysisn 
data.drop("Id", axis=1, inplace=True)
data.head(3)

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500


In [50]:
# Checking missing values
data.isnull().sum()

MSSubClass         0
MSZoning           0
LotFrontage      259
LotArea            0
Street             0
                ... 
MoSold             0
YrSold             0
SaleType           0
SaleCondition      0
SalePrice          0
Length: 80, dtype: int64

In [51]:
# Visualization of the first 3 rows of the dataset after removing the 'customer_id' column
data.head(3)

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500


In [52]:
# Checking for duplicates
print(data.duplicated().sum())

0


In [53]:
# Checking categorial columns and let's look at the unique values
categorical_cols = data.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns:", categorical_cols)

Categorical columns: ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']


In [54]:
# Deleting outliers based on 'LotArea' and 'GarageArea' columns
data = data[data['LotArea'] < 100000]
data = data[data['GarageArea'] < 1400]
print(f"Shape après outliers: {data.shape}")

Shape après outliers: (1455, 80)


In [55]:
# Let's drop the features with more than 50% missing values
threshold = 0.5
missing_percentages = data.isnull().mean()
columns_to_drop = missing_percentages[missing_percentages > threshold].index
data.drop(columns=columns_to_drop, inplace=True)
print(f"Dropped columns: {columns_to_drop.tolist()}")
categorical_cols = data.select_dtypes(include=['object']).columns.tolist()

Dropped columns: ['Alley', 'MasVnrType', 'PoolQC', 'Fence', 'MiscFeature']


In [56]:
# Let's drop again features with little impact on the target variable
data = data.drop(columns=['BsmtFinSF2', 'BsmtHalfBath', 'MiscVal', 'MoSold'])
data.head(3)


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,YrSold,SaleType,SaleCondition,SalePrice
0,60,RL,65.0,8450,Pave,Reg,Lvl,AllPub,Inside,Gtl,...,0,61,0,0,0,0,2008,WD,Normal,208500
1,20,RL,80.0,9600,Pave,Reg,Lvl,AllPub,FR2,Gtl,...,298,0,0,0,0,0,2007,WD,Normal,181500
2,60,RL,68.0,11250,Pave,IR1,Lvl,AllPub,Inside,Gtl,...,0,42,0,0,0,0,2008,WD,Normal,223500


In [57]:
# Multicollinearity correction
data = data.drop(columns=['GarageCars', 'TotRmsAbvGrd'])

In [58]:
# Imputing missing values for numerical columns with the median
numerical_cols = data.select_dtypes(include=['int64', 'float64']).columns.tolist()
for col in numerical_cols:
    data[col] = data[col].fillna(data[col].median())

In [59]:
# Imputing missing values for categorical columns with the mode
for col in categorical_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

In [60]:
# Transforming target variable 'SalePrice' with log transformation to reduce skewness
data['SalePrice'] = np.log1p(data['SalePrice'])
data['SalePrice'].head(3)

0    12.247699
1    12.109016
2    12.317171
Name: SalePrice, dtype: float64

In [61]:
# Encoding categorical variables using One-Hot Encoding
encoder = OneHotEncoder(sparse_output=False, drop='first')
encoded_cols = encoder.fit_transform(data[categorical_cols])
encoded_col_names = encoder.get_feature_names_out(categorical_cols)
encoded_df = pd.DataFrame(encoded_cols, columns=encoded_col_names, index=data.index)
data = pd.concat([data.drop(columns=categorical_cols), encoded_df], axis=1)

In [62]:
# Feature engineering

# Surface totale de la maison
data['TotalSF'] = data['TotalBsmtSF'] + data['1stFlrSF'] + data['2ndFlrSF']

# Age de la maison au moment de la vente
data['HouseAge'] = data['YrSold'] - data['YearBuilt']

# Années depuis la rénovation
data['YearsSinceRemodel'] = data['YrSold'] - data['YearRemodAdd']

# Surface totale des terrasses/porches
data['TotalPorchSF'] = data['OpenPorchSF'] + data['EnclosedPorch'] + data['ScreenPorch'] + data['3SsnPorch'] + data['WoodDeckSF']

# Nombre total de salles de bains
data['TotalBathrooms'] = data['FullBath'] + data['HalfBath'] * 0.5 + data['BsmtFullBath']

# La maison a-t-elle un garage ?
data['HasGarage'] = (data['GarageArea'] > 0).astype(int)

# La maison a-t-elle une piscine ?
data['HasPool'] = (data['PoolArea'] > 0).astype(int)

# La maison a-t-elle une cheminée ?
data['HasFireplace'] = (data['Fireplaces'] > 0).astype(int)

# La maison a-t-elle été rénovée ?
data['IsRemodeled'] = (data['YearBuilt'] != data['YearRemodAdd']).astype(int)

In [63]:
# Let's drop the original columns that we used to create the new features
data = data.drop(columns=[
    '1stFlrSF', '2ndFlrSF', 'TotalBsmtSF', 'OpenPorchSF', 'EnclosedPorch', 'ScreenPorch', 
    '3SsnPorch', 'WoodDeckSF', 'FullBath', 'HalfBath', 'BsmtFullBath',     'YearBuilt', 'YearRemodAdd'                 
])

In [64]:
# Correlation of the target variable 'SalePrice' with only new features
new_features = ['TotalSF', 'HouseAge', 'YearsSinceRemodel', 'TotalPorchSF', 'TotalBathrooms', 'HasGarage', 'HasPool', 'HasFireplace', 'IsRemodeled']
correlations = data[new_features + ['SalePrice']].corr()['SalePrice'].sort_values(ascending=False)
print(correlations)

SalePrice            1.000000
TotalSF              0.812708
TotalBathrooms       0.668852
HasFireplace         0.508744
TotalPorchSF         0.400969
HasGarage            0.323072
HasPool              0.076977
IsRemodeled         -0.074305
YearsSinceRemodel   -0.571030
HouseAge            -0.590267
Name: SalePrice, dtype: float64


In [65]:
# Visualization of the first 3 rows of the dataset after processing
data.head()

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,MasVnrArea,BsmtFinSF1,BsmtUnfSF,LowQualFinSF,GrLivArea,...,SaleCondition_Partial,TotalSF,HouseAge,YearsSinceRemodel,TotalPorchSF,TotalBathrooms,HasGarage,HasPool,HasFireplace,IsRemodeled
0,60,65.0,8450,7,5,196.0,706,150,0,1710,...,0.0,2566,5,5,61,3.5,1,0,0,0
1,20,80.0,9600,6,8,0.0,978,284,0,1262,...,0.0,2524,31,31,298,2.0,1,0,1,0
2,60,68.0,11250,7,5,162.0,486,434,0,1786,...,0.0,2706,7,6,42,3.5,1,0,1,1
3,70,60.0,9550,7,5,0.0,216,540,0,1717,...,0.0,2473,91,36,307,2.0,1,0,1,1
4,60,84.0,14260,8,5,350.0,655,490,0,2198,...,0.0,3343,8,8,276,3.5,1,0,1,0


In [66]:
# Verify missing values
print(data.isnull().sum())

MSSubClass        0
LotFrontage       0
LotArea           0
OverallQual       0
OverallCond       0
                 ..
TotalBathrooms    0
HasGarage         0
HasPool           0
HasFireplace      0
IsRemodeled       0
Length: 223, dtype: int64


In [67]:
# Save train
data.to_csv("../data/processed/train_clean.csv", index=False)
print(f"Shape finale: {data.shape}")
print("Saved!")

Shape finale: (1455, 223)
Saved!


##  Conclusion

### Nettoyage
- Suppression de `Id` (non informatif) et de 5 colonnes avec >50% de valeurs manquantes (`Alley`, `MasVnrType`, `PoolQC`, `Fence`, `MiscFeature`).
- Suppression de 4 features à faible impact sur `SalePrice` : `BsmtFinSF2`, `BsmtHalfBath`, `MiscVal`, `MoSold`.
- Correction de la multicolinéarité : suppression de `GarageCars` (redondant avec `GarageArea`) et `TotRmsAbvGrd` (redondant avec `GrLivArea`).

### Imputation
- Valeurs manquantes numériques imputées par la **médiane**.
- Valeurs manquantes catégorielles imputées par le **mode**.
- 0 valeurs manquantes après imputation 

### Transformation
- `SalePrice` transformé en `log1p` pour corriger l'asymétrie de la distribution.

### Encodage
- 43 variables catégorielles encodées avec **One-Hot Encoding** (`drop='first'` pour éviter la multicolinéarité).

### Feature Engineering
- 9 nouvelles features créées : `TotalSF`, `HouseAge`, `YearsSinceRemodel`, `TotalPorchSF`, `TotalBathrooms`, `HasGarage`, `HasPool`, `HasFireplace`, `IsRemodeled`.
- `TotalSF` est la feature la plus corrélée avec `SalePrice` parmi les nouvelles features (0.78).
- `HouseAge` et `YearsSinceRemodel` montrent une corrélation négative attendue (-0.59 et -0.57).

### Dataset final
- Shape finale : **(1460, 224)** — prêt pour la modélisation.